# C4-filtered: pre-registered selection condition

Evaluates synthetic df images selected by distance to real df, using the rules in `C4_FILTERED_EXPERIMENT_DESIGN.md`.

The earlier gate used epoch-60 percentiles (`p1 = 0.6564`, `p5 = 0.8293`). The published epoch-100 pool has `p25 = 0.8508` and `p50 = 0.9990` against the unchanged threshold `0.901291`, placing the estimated accepted count between 125 and 250 of 500 images. The acceptance script determines the actual count; fewer than 50 blocks training.

This condition evaluates the test split once; Phase 4 rejects an existing run record. The test split contains 16 real df images. Results are descriptive, and no significance test is performed.


In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "e68b3a79387c91e9c4913ac363cf41297e4db83b"
assert len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH", "Pin the reviewed pushed commit before Run all"
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
RUN_VERSION = "v1_c4_filtered"

# Held fixed to the formal matched-585 run, read from its own results JSON.
# C4-filtered must differ from C1 and C4 in the df source and nothing else.
SEEDS = (0, 1, 2)
EPOCHS = 20
DF_TARGET_COUNT = 585
BATCH_SIZE = 32
IMG_SIZE = 128
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SYNTHETIC_POOL_DIR = SHARED_PROJECT_DIR / "outputs" / "synthetic_df" / "epoch0100_seed0"
CANDIDATE_MANIFEST = SYNTHETIC_POOL_DIR / "synthetic_df.csv"
JUDGE_CHECKPOINT = SHARED_PROJECT_DIR / "outputs" / "classifier_df585" / "checkpoints" / "C1_seed2" / "best.pt"
FORMAL_RESULTS_DIR = SHARED_PROJECT_DIR / "outputs" / "classifier_df585" / "results"
RUN_ROOT = SHARED_PROJECT_DIR / "outputs" / "c4_filtered" / RUN_VERSION
ACCEPTED_MANIFEST = RUN_ROOT / "c4_filtered_accepted.csv"
SELECTION_RECORD = RUN_ROOT / "c4_filtered_selection.json"
CODE_DIR = Path("/content/ddpm-code")

## Phase 0 — mount Drive, clone the pinned commit, install dependencies

In [ ]:
import base64, json, os, subprocess, sys, time
from google.colab import drive, userdata
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project folder: {SHARED_PROJECT_DIR}"
assert CANDIDATE_MANIFEST.is_file(), f"missing candidate pool manifest: {CANDIDATE_MANIFEST}"
assert (SYNTHETIC_POOL_DIR / "_READY.json").is_file(), f"pool is not the published one: {SYNTHETIC_POOL_DIR}"
assert JUDGE_CHECKPOINT.is_file(), f"missing C1 judge checkpoint: {JUDGE_CHECKPOINT}"

token = userdata.get("GH_TOKEN")
assert token and len(token) > 20, "Colab Secret GH_TOKEN with read access to this repo is required"
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
basic_credential = base64.b64encode(("x-access-token:" + token).encode()).decode()
clone_env = os.environ.copy()
clone_env["GIT_TERMINAL_PROMPT"] = "0"
clone_env["GIT_CONFIG_COUNT"] = "1"
clone_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
clone_env["GIT_CONFIG_VALUE_0"] = "Authorization: Basic " + basic_credential
try:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True, env=clone_env)
finally:
    clone_env["GIT_CONFIG_VALUE_0"] = ""
    token = basic_credential = None
    del token, basic_credential, clone_env
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
remote = subprocess.check_output(["git", "-C", str(CODE_DIR), "remote", "get-url", "origin"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status, "clone must be a clean detached checkout of the pinned commit"
assert "@" not in remote and "x-access-token" not in remote, "clone URL must not embed a credential"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas>=2.0", "pillow>=9.0"], check=True)

import torch
assert torch.cuda.is_available(), "a GPU runtime is required; Runtime -> Change runtime type -> GPU"
print(json.dumps({"commit": commit, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__}, indent=2))

# The scripts resolve the fixed split through the repository's own config.
os.environ["DDPM_DERM_DATA_DIR"] = str(SHARED_PROJECT_DIR / "data")
os.environ["DDPM_DERM_OUTPUTS_DIR"] = str(SHARED_PROJECT_DIR / "outputs")
env = os.environ.copy()
env["PYTHONPATH"] = str(CODE_DIR / "src")
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONDONTWRITEBYTECODE"] = "1"

## Phase 1: Streaming subprocess output


In [ ]:
import queue, threading

def run_stream(command, cwd=CODE_DIR, heartbeat=60):
    started = time.monotonic()
    process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines, output_queue = [], queue.Queue()
    def pump():
        for line in process.stdout: output_queue.put(line)
        output_queue.put(None)
    threading.Thread(target=pump, daemon=True).start()
    while True:
        try: line = output_queue.get(timeout=heartbeat)
        except queue.Empty:
            print(f"[subprocess] heartbeat elapsed={time.monotonic()-started:.0f}s alive={process.poll() is None}", flush=True)
            continue
        if line is None: break
        lines.append(line); print(line, end="", flush=True)
    code = process.wait()
    if code: raise subprocess.CalledProcessError(code, command)
    return time.monotonic() - started, "".join(lines)

## Phase 2 — the selection

Applies the acceptance rule and nothing else. The judge is the C1 seed 2
ResNet-18, which is trained on real images only and so never saw the pool it is
filtering. The threshold is recomputed here from the 14 val `df`, never
hardcoded.

The accepted manifest is written into the run root rather than into the
published pool directory, so that directory stays exactly as it was; the images
are still resolved from it through `--generated-root`.

In [ ]:
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# Run-all-safe. The selection rule is deterministic and its record carries the
# hash of the pool it measured, so an existing complete record is reused rather
# than recomputed. Only a *partial* record is refused, because that is the one
# case where what is on disk cannot be trusted to describe a finished selection.
if SELECTION_RECORD.exists():
    existing = json.loads(SELECTION_RECORD.read_text(encoding="utf-8"))
    assert "condition_runnable" in existing, (
        f"a partial selection record exists; move it aside deliberately: {SELECTION_RECORD}"
    )
    assert existing["candidate_manifest"].endswith(CANDIDATE_MANIFEST.name), (
        f"the existing record measured {existing['candidate_manifest']}, not "
        f"{CANDIDATE_MANIFEST}; move it aside deliberately"
    )
    if existing["condition_runnable"]:
        assert ACCEPTED_MANIFEST.is_file(), (
            f"the record accepts {existing['accepted_count']} images but "
            f"{ACCEPTED_MANIFEST} is missing; move both aside and reselect"
        )
    print(f"[select] reusing the existing selection: {SELECTION_RECORD}")
    print(f"  accepted {existing['accepted_count']} of {existing['counts']['candidate']}")
    print(f"  pool measured: {existing['synthetic_provenance'].get('content_sha256')}")
else:
    elapsed, _ = run_stream([
        sys.executable, "-B", "-u", "scripts/c4_filtered_select.py",
        "--candidate-manifest", str(CANDIDATE_MANIFEST),
        "--judge-checkpoint", str(JUDGE_CHECKPOINT),
        "--accepted-out", str(ACCEPTED_MANIFEST),
        "--record-out", str(SELECTION_RECORD),
    ])
    print("")
    print(f"[select] COMPLETE in {elapsed:.0f}s -> {SELECTION_RECORD}")


## Phase 3: Minimum sample count

The pre-registered minimum is 50 accepted images. Below this count, the selection script writes no accepted manifest and training is blocked.


In [ ]:
selection = json.loads(SELECTION_RECORD.read_text(encoding="utf-8"))
threshold = selection["threshold"]
accepted = selection["accepted_count"]
print(f"threshold  max(val df -> train) = {threshold:.6f}")
print(f"accepted   {accepted} of {selection['counts']['candidate']} "
      f"({selection['accepted_fraction']:.1%} of the pool)")
print(f"minimum    {selection['minimum_accepted']}  ->  runnable={selection['condition_runnable']}")

if "accepted_to_train" in selection["distances"]:
    a = selection["distances"]["accepted_to_train"]
    print(f"\naccepted -> train distance: min={a['p0']:.4f} median={a['p50']:.4f} max={a['p100']:.4f}")
if "diversity" in selection:
    d = selection["diversity"]
    print(f"within-set spacing, accepted / real = {d['within_set_median_ratio']:.3f}")
    print("  (<1 means filtering bought closeness by collapsing variety; report it either way)")

assert selection["condition_runnable"], (
    f"only {accepted} images cleared the threshold, below the pre-registered "
    f"minimum of {selection['minimum_accepted']}. The condition is not run and "
    "this shortfall is the result. Record it and stop here."
)
assert ACCEPTED_MANIFEST.is_file(), f"accepted manifest missing: {ACCEPTED_MANIFEST}"
print("\n[gate] cleared -- training may start")

## Phase 4 — train the condition

Three seeds, 20 epochs, identical to the formal matched-585 runs in everything
except where the df rows come from. `df_target_count` stays 585: the accepted
images go in, and the remaining slots are filled by real duplication, which is
exactly what C1 does.

**This evaluates the test split.** The assert below is what keeps that to a
single occasion.

In [ ]:
results_dir = RUN_ROOT / "results"
assert not (results_dir / f"results_C4_FILTERED_seed{SEEDS[-1]}.json").exists(), (
    "a completed C4-filtered run already exists under this run version. The "
    "design evaluates the test split once; start a new run version deliberately "
    "rather than overwriting this one."
)

for seed in SEEDS:
    print(f"\n{'='*60}\n[train] C4_FILTERED seed {seed}\n{'='*60}", flush=True)
    elapsed, _ = run_stream([
        sys.executable, "-B", "-u", "-m", "ddpm_derm.train_classifier",
        "--variant", "C4_FILTERED",
        "--accepted-manifest", str(ACCEPTED_MANIFEST),
        "--generated-root", str(SYNTHETIC_POOL_DIR),
        "--seed", str(seed),
        "--epochs", str(EPOCHS),
        "--df-target-count", str(DF_TARGET_COUNT),
        "--batch-size", str(BATCH_SIZE),
        "--img-size", str(IMG_SIZE),
        "--lr", str(LEARNING_RATE),
        "--weight-decay", str(WEIGHT_DECAY),
        "--num-workers", str(NUM_WORKERS),
        "--output-dir", str(RUN_ROOT),
        "--resume",
    ])
    print(f"[train] seed {seed} COMPLETE in {elapsed/60:.1f} min", flush=True)

## Phase 5: Results

Reads C1 and C4 comparison values from their recorded result files. Section 6 of the design defines the interpretation:

- Above both C1 and C4: a descriptive gain under the filtered condition.
- Near C1: no clear improvement over real-image duplication.
- Near C4: no clear improvement from filtering.

Comparisons use the fixed test split with 16 df images and do not establish statistical significance.


In [ ]:
import statistics

def read_condition(directory, variant):
    values = {}
    for seed in SEEDS:
        path = Path(directory) / f"results_{variant}_seed{seed}.json"
        if not path.is_file():
            return None
        payload = json.loads(path.read_text(encoding="utf-8"))
        values[seed] = payload["test_metrics"]
    return values

def summarise(name, per_seed):
    if per_seed is None:
        print(f"{name:14} (not found)")
        return None
    f1 = [per_seed[s]["target_f1"] for s in SEEDS]
    macro = [per_seed[s]["macro_f1"] for s in SEEDS]
    recall = [per_seed[s]["target_recall"] for s in SEEDS]
    mean = statistics.fmean(f1)
    print(f"{name:14} df F1 {mean:.4f} +/- {statistics.pstdev(f1):.4f}   "
          f"macro {statistics.fmean(macro):.4f}   df recall {statistics.fmean(recall):.4f}")
    return mean

print("test split, mean +/- population std over seeds 0/1/2, df support = 16\n")
c1 = summarise("C1@585", read_condition(FORMAL_RESULTS_DIR, "C1"))
c4 = summarise("C4@585", read_condition(FORMAL_RESULTS_DIR, "C4"))
filtered = summarise("C4-filtered", read_condition(RUN_ROOT / "results", "C4_FILTERED"))

if filtered is not None:
    print(f"\naccepted {selection['accepted_count']} of {selection['counts']['candidate']} "
          f"synthetic images; the other {DF_TARGET_COUNT - 85 - selection['accepted_count']} "
          "df slots are duplicated real images")
    if c1 is not None:
        print(f"C4-filtered - C1  = {filtered - c1:+.4f}")
    if c4 is not None:
        print(f"C4-filtered - C4  = {filtered - c4:+.4f}")
    print("\nDescriptive only. 16 test df; no significance test; the procedure and "
          "the measured distances are the deliverable, not the delta.")